# Notebook 02 — Modélisation : prévision et profils de journées
**Projet final · S7MOI1 · Science des Données, concepts avancés · ECAM-EPMI · 2AE**
Équipe : AKONDE Esly · DELBARD-MAZZOLA Paul-Baptiste · DIOP Abdou Aziz

## Point de départ
Le notebook 01 a produit trois fichiers dans `data/` :
- `features_journalieres.csv` : la cible `conso_kwh` et les variables causales (`lag_1`, `lag_7`, `mm7`, calendrier) ;
- `table_journaliere.csv` : la table journalière complète ;
- `profils_horaires.csv` : 1 417 journées × 24 heures (puissance moyenne par heure).

Résultats du notebook 01 utilisés ici : consommation moyenne de 26,2 kWh/jour ; baselines évaluées sur 2009 : « comme hier » = 6,29 kWh/jour, « semaine dernière » = 5,77 kWh/jour ; objectif de succès : 4,90 kWh/jour (−15 %).

## Démarche
**Partie A — Problématique 1 (prévision, supervisée) :** split chronologique, baselines, régression linéaire, arbre et forêt aléatoire, réglage d'un hyperparamètre, diagnostic, évaluation finale sur le test (une seule fois).

**Partie B — Problématique 2 (non supervisée) :** standardisation, PCA, K-Means (coude, silhouette), DBSCAN, description des profils de journées, lien avec les erreurs de prévision.

**Principes appliqués :** split avant toute préparation, préparation incluse dans un `Pipeline`, baseline à battre, validation croisée avec moyenne et écart-type, jeu de test utilisé une seule fois.

## 0 · Mise en place et chargement
Si la session Colab a été réinitialisée, le dossier `data/` est vide : relancer le notebook 01 (*Exécution → Tout exécuter*) ou importer les trois fichiers CSV lorsque la cellule le demande.

In [ ]:
import os
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from sklearn.base import clone
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.linear_model import LinearRegression
from sklearn.tree import DecisionTreeRegressor
from sklearn.ensemble import RandomForestRegressor
from sklearn.model_selection import TimeSeriesSplit, GridSearchCV, cross_validate
from sklearn.decomposition import PCA
from sklearn.cluster import KMeans, DBSCAN
from sklearn.neighbors import NearestNeighbors
from sklearn.metrics import (mean_absolute_error, mean_squared_error,
                             silhouette_score, adjusted_rand_score)

SEED = 42                       # graine fixe : résultats reproductibles
np.random.seed(SEED)
plt.rcParams["figure.figsize"] = (10, 4)

DATA_DIR = "data"
FICHIERS = ["features_journalieres.csv", "table_journaliere.csv", "profils_horaires.csv"]

# Les CSV sont lus localement s'ils existent. Sinon, on les télécharge depuis le dossier data/ du dépôt GitHub.
os.makedirs(DATA_DIR, exist_ok=True)
BASE_CSV_URL = "https://raw.githubusercontent.com/akondeesly-cyber/prevision-conso-electrique/main/data/"
for nom in FICHIERS:
    chemin = os.path.join(DATA_DIR, nom)
    if not os.path.exists(chemin):
        url = BASE_CSV_URL + nom
        try:
            print(f"Téléchargement de {nom} depuis GitHub...")
            pd.read_csv(url, nrows=1)  # vérifie que le fichier est accessible
            import urllib.request
            urllib.request.urlretrieve(url, chemin)
        except Exception as e:
            raise FileNotFoundError(
                f"Impossible de récupérer {nom} depuis {url}. "
                "Vérifiez que le CSV a été ajouté au dossier data/ du dépôt GitHub. "
                "En local, exécutez d'abord le notebook précédent."
            ) from e

manquants = [f for f in FICHIERS if not os.path.exists(os.path.join(DATA_DIR, f))]
if manquants:
    raise FileNotFoundError(f"Fichiers CSV manquants : {manquants}")

feat = pd.read_csv(os.path.join(DATA_DIR, "features_journalieres.csv"), index_col="date", parse_dates=True)
table_j = pd.read_csv(os.path.join(DATA_DIR, "table_journaliere.csv"), index_col="date", parse_dates=True)
profils = pd.read_csv(os.path.join(DATA_DIR, "profils_horaires.csv"), index_col="date", parse_dates=True)

print("features_journalieres :", feat.shape)
print("table_journaliere     :", table_j.shape)
print("profils_horaires      :", profils.shape)

# PARTIE A — Prévision de la consommation de demain

## A1 · Split chronologique
**Objectif :** on sépare le passé (entraînement) du futur (test) **par une date de coupure**.

**Pourquoi :** en prévision, on ne mélange jamais les dates (CM04). Si on tirait les jours au hasard, le modèle s'entraînerait sur des jours *après* ceux qu'il doit prévoir : le futur servirait à prédire le passé.

**Choix :**
- **Train** : du début jusqu'au 31/12/2009.
- **Test** : 2010 (la période la plus récente), mis de côté jusqu'à la section A5.
- On ne garde que les lignes où la cible **et** toutes les variables sont connues (les lignes incomplètes ont été identifiées dans le notebook 01).

In [ ]:
FEATURES = ["lag_1", "lag_7", "mm7", "est_weekend", "jour_semaine", "mois"]
CIBLE = "conso_kwh"

dispo = feat.dropna(subset=[CIBLE] + FEATURES)
DATE_COUPURE = pd.Timestamp("2010-01-01")

train = dispo.loc[dispo.index < DATE_COUPURE]
test = dispo.loc[dispo.index >= DATE_COUPURE]
X_train, y_train = train[FEATURES], train[CIBLE]
X_test, y_test = test[FEATURES], test[CIBLE]

assert train.index.max() < test.index.min(), "Le train doit précéder le test !"
print(f"Train : {len(train)} jours ({train.index.min().date()} -> {train.index.max().date()})")
print(f"Test  : {len(test)} jours ({test.index.min().date()} -> {test.index.max().date()})")

fig, ax = plt.subplots(figsize=(12, 3.5))
ax.plot(train.index, y_train, linewidth=0.6, label="Train")
ax.plot(test.index, y_test, linewidth=0.6, color="tab:orange", label="Test (non touché jusqu'à A5)")
ax.axvline(DATE_COUPURE, color="black", linestyle="--")
ax.set_title("Découpage chronologique train / test")
ax.set_xlabel("Date"); ax.set_ylabel("Énergie (kWh par jour)")
ax.legend(); plt.tight_layout(); plt.show()

## A2 · Pipeline, baselines et premiers modèles
**Objectif :** construire un `Pipeline` (préparation + modèle) et comparer plusieurs modèles par **validation croisée temporelle** (`TimeSeriesSplit`, 5 découpes).

**Comment fonctionne `TimeSeriesSplit` :** à chaque découpe, on entraîne sur le passé et on valide sur la période juste après. Le train grossit à chaque pli, la validation est toujours *dans le futur* du train.

**Préparation (dans le Pipeline) :**
- `lag_1`, `lag_7`, `mm7` : standardisés (`StandardScaler`, ajusté sur le train de chaque pli seulement) ;
- `jour_semaine`, `mois` : encodés en indicatrices (`OneHotEncoder`) ; ce sont des catégories, pas des quantités ;
- `est_weekend` : laissé tel quel (déjà 0/1).

**Modèles comparés :**
1. **Baselines** : « comme hier » (`lag_1`) et « comme la semaine dernière » (`lag_7`).
2. **Régression linéaire** : modèle simple et explicable.
3. **Arbre sans limite de profondeur** : pour **voir le surapprentissage** (CM01).
4. **Random Forest** : modèle à base d'arbres, moyenne de nombreux arbres.

**Métrique principale :** MAE en kWh par jour (erreur moyenne dans l'unité métier, axe 5).

In [ ]:
NUM = ["lag_1", "lag_7", "mm7"]
CAT = ["jour_semaine", "mois"]
BIN = ["est_weekend"]

def faire_preprocess():
    return ColumnTransformer([
        ("num", StandardScaler(), NUM),
        ("cat", OneHotEncoder(handle_unknown="ignore", sparse_output=False), CAT),
        ("bin", "passthrough", BIN),
    ])

tscv = TimeSeriesSplit(n_splits=5)

def mae_baseline_cv(colonne):
    # MAE d'une baseline (lag) sur chaque pli de validation, sans aucun apprentissage
    return np.array([mean_absolute_error(y_train.iloc[va], X_train[colonne].iloc[va])
                     for _, va in tscv.split(X_train)])

modeles = {
    "Régression linéaire": Pipeline([("prep", faire_preprocess()), ("m", LinearRegression())]),
    "Arbre (sans max_depth)": Pipeline([("prep", faire_preprocess()),
                                        ("m", DecisionTreeRegressor(random_state=SEED))]),
    "Random Forest (défaut)": Pipeline([("prep", faire_preprocess()),
                                        ("m", RandomForestRegressor(n_estimators=200, random_state=SEED, n_jobs=-1))]),
}

lignes = []
for nom, col in [("Baseline naïve (hier)", "lag_1"), ("Baseline saisonnière (sem. passée)", "lag_7")]:
    s = mae_baseline_cv(col)
    lignes.append([nom, np.nan, s.mean(), s.std()])

for nom, pipe in modeles.items():
    cv = cross_validate(pipe, X_train, y_train, cv=tscv,
                        scoring="neg_mean_absolute_error", return_train_score=True)
    lignes.append([nom, -cv["train_score"].mean(), -cv["test_score"].mean(), cv["test_score"].std()])

tab = pd.DataFrame(lignes, columns=["Modèle", "MAE train", "MAE validation (moy.)", "Écart-type"])
meilleure_base_cv = tab.loc[tab["Modèle"].str.startswith("Baseline"), "MAE validation (moy.)"].min()
tab["Gain vs meilleure baseline"] = (1 - tab["MAE validation (moy.)"] / meilleure_base_cv).map("{:.1%}".format)
tab.round(2)

**Interprétation.** En validation croisée temporelle, la meilleure baseline est « comme hier » (MAE 6,18 kWh/jour, écart-type 0,69), devant « même jour la semaine dernière » (6,40 ; soit 3,5 % de plus). Ce classement est l'inverse de celui observé sur la seule année 2009 dans le notebook 01 (5,77 pour la semaine dernière contre 6,29 pour hier) : le classement des baselines dépend de la période, et nous retenons ici la moyenne sur cinq périodes de validation.

La régression linéaire bat les baselines : MAE de 5,32 kWh/jour, soit un gain de 13,9 %, mais avec une dispersion élevée d'un pli à l'autre (écart-type 0,95).

L'arbre sans limite de profondeur illustre le surapprentissage : erreur nulle sur le train (0,00) et 8,22 en validation, soit 33 % de plus que la meilleure baseline. Le Random Forest par défaut corrige en grande partie ce défaut en moyennant de nombreux arbres, ce qui réduit la variance (validation 5,67, gain de 8,3 %), mais il surapprend encore (train 2,27 contre 5,67 en validation).

## A3 · Réglage d'un hyperparamètre (Random Forest)
**Objectif :** chercher les meilleures valeurs de `max_depth` (profondeur maximale des arbres) et `min_samples_leaf` (taille minimale d'une feuille) avec `GridSearchCV`.

**Pourquoi :** une forêt trop profonde mémorise le train (surapprentissage), une forêt trop peu profonde ne capture rien (sous-apprentissage). On cherche le bon compromis **par validation croisée temporelle sur le train**, jamais sur le test.

**Point technique :** `GridSearchCV` ré-entraîne ensuite automatiquement le meilleur modèle sur *tout* le train (`refit=True`).

In [ ]:
rf_pipe = Pipeline([("prep", faire_preprocess()),
                    ("rf", RandomForestRegressor(n_estimators=200, random_state=SEED, n_jobs=-1))])
grille = {"rf__max_depth": [3, 5, 8, 12],
          "rf__min_samples_leaf": [1, 5, 10]}

gs = GridSearchCV(rf_pipe, grille, cv=tscv, scoring="neg_mean_absolute_error", n_jobs=-1)
gs.fit(X_train, y_train)

res = pd.DataFrame(gs.cv_results_)
res["MAE validation"] = -res["mean_test_score"]
res["Écart-type"] = res["std_test_score"]
print("Meilleurs hyperparamètres :", gs.best_params_)
res[["param_rf__max_depth", "param_rf__min_samples_leaf", "MAE validation", "Écart-type"]] \
    .sort_values("MAE validation").head(6).round(2)

In [ ]:
# Ajout du Random Forest réglé au tableau de comparaison
best_rf = gs.best_estimator_
cv_best = cross_validate(clone(best_rf), X_train, y_train, cv=tscv,
                         scoring="neg_mean_absolute_error", return_train_score=True)
ligne = ["Random Forest (réglée)", -cv_best["train_score"].mean(),
         -cv_best["test_score"].mean(), cv_best["test_score"].std()]
tab.loc[len(tab)] = ligne + [""]
tab["Gain vs meilleure baseline"] = (1 - tab["MAE validation (moy.)"] / meilleure_base_cv).map("{:.1%}".format)
tab.round(2)

**Interprétation.** Le réglage fait apparaître un plateau : les profondeurs 5, 8 et 12 avec `min_samples_leaf = 10` donnent la même MAE de validation (5,23 kWh/jour, écart-type de 0,58 à 0,60), et la profondeur 3 est un peu moins bonne (5,31). Le paramètre qui compte surtout est `min_samples_leaf` : exiger au moins 10 jours par feuille empêche la forêt de mémoriser le train. `GridSearchCV` retient `max_depth = 12` et `min_samples_leaf = 10`, un réglage parmi plusieurs équivalents.

Le surapprentissage disparaît : MAE de 5,19 sur le train contre 5,23 en validation (contre 2,27 et 5,67 pour la forêt par défaut). La forêt réglée atteint 5,23 ± 0,60 kWh/jour, soit un gain de 15,4 % sur la meilleure baseline. Elle est plus stable d'un pli à l'autre que la régression linéaire (écart-type 0,60 contre 0,95), même si l'écart de MAE entre les deux modèles est faible (5,23 contre 5,32) : la régression linéaire, plus simple et explicable, reste un concurrent sérieux.

## A4 · Diagnostic : variables importantes et analyse des erreurs
**Objectif :**
1. Regarder quelles variables le Random Forest utilise le plus (`feature_importances_`).
2. Calculer des **prédictions « hors échantillon »** : pour chaque jour de validation, la prédiction faite par un modèle qui n'avait vu que le passé. On analyse les erreurs **sans toucher au test**.
3. Chercher *où* le modèle se trompe : par jour de semaine, par mois, les pires jours.
4. Estimer un **intervalle d'erreur à 90 %** à partir de ces erreurs (CM04).

In [ ]:
# 1. Importance des variables
noms = best_rf.named_steps["prep"].get_feature_names_out()
imp = pd.Series(best_rf.named_steps["rf"].feature_importances_, index=noms).sort_values()
fig, ax = plt.subplots(figsize=(8, 5))
imp.tail(10).plot(kind="barh", ax=ax)
ax.set_title("Importance des variables (Random Forest réglée)")
ax.set_xlabel("Importance (part de la réduction d'erreur)")
plt.tight_layout(); plt.show()
print(imp.sort_values(ascending=False).head(6).round(3))

**Interprétation.** La variable dominante est `mm7`, la moyenne des 7 jours précédents (importance 0,60), suivie de `lag_1` (0,21), de `est_weekend` (0,07) et de `lag_7` (0,06). Les variables détaillées de calendrier (jour de la semaine, mois) pèsent chacune moins de 0,01. C'est cohérent avec l'analyse exploratoire : le niveau récent de consommation, qui porte à la fois la saison et les périodes d'absence, explique l'essentiel, et le week-end apporte une correction supplémentaire.

Précaution : `lag_1`, `lag_7` et `mm7` sont corrélées entre elles, ce qui répartit leur importance de façon peu stable. Ces valeurs indiquent ce que la forêt utilise, pas des effets causaux.

In [ ]:
# 2. Prédictions hors échantillon (validation temporelle) : seulement des jours de validation
def predictions_oof(modele, X, y):
    p = pd.Series(np.nan, index=y.index)
    for tr, va in tscv.split(X):
        m = clone(modele).fit(X.iloc[tr], y.iloc[tr])
        p.iloc[va] = m.predict(X.iloc[va])
    return p

oof_rf = predictions_oof(best_rf, X_train, y_train)
err = pd.DataFrame({"reel": y_train, "pred_rf": oof_rf, "pred_lag7": X_train["lag_7"]}).dropna()
err["abs_rf"] = (err["reel"] - err["pred_rf"]).abs()
err["abs_lag7"] = (err["reel"] - err["pred_lag7"]).abs()
print("Jours de validation analysés :", len(err))
print(f"MAE Random Forest : {err['abs_rf'].mean():.2f} | MAE baseline sem. passée : {err['abs_lag7'].mean():.2f} kWh/jour")

# 3. Où le modèle se trompe-t-il ?
jours = ["Lun", "Mar", "Mer", "Jeu", "Ven", "Sam", "Dim"]
par_jour = err.groupby(err.index.dayofweek)[["abs_rf", "abs_lag7"]].mean()
par_mois = err.groupby(err.index.month)[["abs_rf", "abs_lag7"]].mean()

fig, axes = plt.subplots(1, 2, figsize=(12, 4))
x = np.arange(len(par_jour)); w = 0.4
axes[0].bar(x - w/2, par_jour["abs_rf"], w, label="Random Forest")
axes[0].bar(x + w/2, par_jour["abs_lag7"], w, label="Baseline sem. passée")
axes[0].set_xticks(x); axes[0].set_xticklabels([jours[i] for i in par_jour.index])
axes[0].set_title("Erreur moyenne par jour de la semaine"); axes[0].set_ylabel("MAE (kWh par jour)"); axes[0].legend()
x2 = np.arange(len(par_mois))
axes[1].bar(x2 - w/2, par_mois["abs_rf"], w, label="Random Forest")
axes[1].bar(x2 + w/2, par_mois["abs_lag7"], w, label="Baseline sem. passée")
axes[1].set_xticks(x2); axes[1].set_xticklabels(par_mois.index)
axes[1].set_title("Erreur moyenne par mois"); axes[1].set_xlabel("Mois (1 = janvier)"); axes[1].set_ylabel("MAE (kWh par jour)")
plt.tight_layout(); plt.show()

print("Les 8 plus grosses erreurs du Random Forest (validation) :")
pires = err.sort_values("abs_rf").tail(8)[["reel", "pred_rf", "abs_rf"]].round(1)
pires

**Interprétation.** Sur les 910 jours de validation, le Random Forest obtient une MAE de 5,23 kWh/jour contre 6,40 pour la baseline « semaine dernière » sur les mêmes jours. Il fait mieux presque partout : l'erreur est plus faible chaque jour de la semaine. Elle est de l'ordre de 4,2 à 5,0 kWh du lundi au vendredi et reste la plus élevée le week-end (environ 6,2 à 6,3 kWh le samedi et le dimanche), jours plus variables (présence, sorties).

Par mois, l'erreur est la plus forte en février (environ 7,0), en novembre (environ 6,6) et en août (environ 6,3), et la plus faible de mai à septembre hors août (environ 3 à 4,3). Août est le seul mois où le modèle fait moins bien que la baseline (environ 6,3 contre 4,3), vraisemblablement à cause des périodes d'absence, dont le début et la fin sont difficiles à anticiper (hypothèse).

Les huit plus grosses erreurs (22 à 30 kWh) sont toutes des ruptures brutales : des journées très basses (5 à 9 kWh) prévues autour de 28 à 34 kWh (29/07/2007, 28/10/2007, 26/10/2008), des journées très hautes (48 à 57 kWh) prévues autour de 22 à 32 kWh (05/11/2008, 23 et 24/11/2008, 24/12/2009), et le 25/11/2007 (23,3 réel contre 46,2 prévu). Cinq de ces huit jours sont des dimanches. Ce sont des événements que la consommation passée ne permet pas d'anticiper (départs, retours, présence de visiteurs), ce que la partie B cherche à documenter.

In [ ]:
# 4. Intervalle d'erreur empirique à 90 % (CM04) : quantiles des erreurs de validation
residus = err["reel"] - err["pred_rf"]
q05, q95 = residus.quantile(0.05), residus.quantile(0.95)
print(f"Erreur (réel - prévu) : 5e centile = {q05:.1f} kWh, 95e centile = {q95:.1f} kWh")
print(f"=> Prévision +/- : environ [{q05:.1f} ; +{q95:.1f}] kWh autour de la valeur prévue, dans 90 % des cas.")

## A5 · Évaluation finale sur le test (une seule fois)
> Le jeu de test n'est évalué **qu'une seule fois**, une fois le choix du modèle terminé : le relancer pour comparer d'autres modèles le transformerait en jeu d'entraînement. Si le modèle est modifié après lecture de ce résultat, nous le signalons dans le rapport.

**Objectif :** le modèle retenu (Random Forest réglé, déjà ré-entraîné sur tout le train) prédit 2010. On compare à la **baseline** sur la même période, puis on vérifie le critère de succès.

In [ ]:
pred_test = pd.Series(best_rf.predict(X_test), index=X_test.index)

mae_rf = mean_absolute_error(y_test, pred_test)
rmse_rf = np.sqrt(mean_squared_error(y_test, pred_test))
mae_naive_t = mean_absolute_error(y_test, X_test["lag_1"])
mae_saison_t = mean_absolute_error(y_test, X_test["lag_7"])
meilleure_base_t = min(mae_naive_t, mae_saison_t)
gain = 1 - mae_rf / meilleure_base_t

print(f"Jours de test : {len(y_test)}")
print(f"MAE baseline naïve (hier)        : {mae_naive_t:.2f} kWh/jour")
print(f"MAE baseline saisonnière (lag 7) : {mae_saison_t:.2f} kWh/jour")
print(f"MAE Random Forest (réglée)       : {mae_rf:.2f} kWh/jour   (RMSE = {rmse_rf:.2f})")
print(f"Gain vs meilleure baseline       : {gain:.1%}  | critère de succès (>= 15 %) atteint ? {gain >= 0.15}")

couverture = ((y_test >= pred_test + q05) & (y_test <= pred_test + q95)).mean()
print(f"Part des jours de test dans l'intervalle à 90 % : {couverture:.1%}")

fig, ax = plt.subplots(figsize=(12, 4))
ax.plot(y_test.index, y_test, linewidth=0.9, label="Réel")
ax.plot(pred_test.index, pred_test, linewidth=0.9, label="Prévu (Random Forest)")
ax.fill_between(pred_test.index, pred_test + q05, pred_test + q95, alpha=0.2, label="Intervalle à 90 %")
ax.set_title("Consommation journalière 2010 : réel vs prévu")
ax.set_xlabel("Date"); ax.set_ylabel("Énergie (kWh par jour)")
ax.legend(); plt.tight_layout(); plt.show()

pd.DataFrame({"reel": y_test, "pred_rf": pred_test, "pred_lag1": X_test["lag_1"], "pred_lag7": X_test["lag_7"]}) \
    .to_csv(os.path.join(DATA_DIR, "predictions_test.csv"), index_label="date")

**Interprétation.** Sur les 305 jours de test (2010), le Random Forest obtient une MAE de 4,22 kWh/jour (RMSE 5,78), contre 5,10 pour « comme hier » et 5,98 pour « semaine dernière ». Le gain par rapport à la meilleure baseline (« comme hier ») est de 17,2 % : **le critère de succès (au moins 15 %) est atteint.** La MAE est aussi inférieure à l'objectif absolu de 4,90 kWh/jour calibré sur 2009. En énergie, le modèle réduit l'erreur moyenne d'environ 0,9 kWh par jour (5,10 → 4,22).

Ce résultat est cohérent avec la validation croisée (gain de 15,4 %). Les erreurs absolues sont plus faibles sur le test que sur la validation (4,22 contre 5,23 pour le modèle, 5,10 contre 6,18 pour la baseline) : 2010 est une année moins perturbée que 2007-2008, qui contiennent les ruptures les plus fortes (section A4). Le gain relatif est donc le chiffre comparable, plus que la MAE brute.

L'intervalle empirique à 90 % estimé sur la validation va de −10,3 à +10,6 kWh autour de la prévision ; il couvre 92,5 % des jours de test. Il est large (environ ±40 % de la consommation moyenne de 26 kWh) : un décideur doit prévoir une marge de sécurité importante. Le modèle semble suivre avec retard les ruptures brutales (creux d'août 2010), ce qui est attendu pour un modèle qui ne connaît que le passé.

Limites : un seul foyer et une seule période de test de 305 jours.

# PARTIE B — Profils types de journées (non supervisé)

**Objectif :** découvrir s'il existe des **types de journées** (ex. journée d'hiver, journée de week-end, journée d'absence), sans étiquette connue au départ (CM02).

> Rappel CM02 : ici il n'y a pas de cible à prédire, donc **pas de split train/test**. L'évaluation est *interne* (inertie, silhouette) et surtout *interprétative* : un cluster n'a de valeur que s'il est explicable et utile au métier.

## B1 · Standardisation
**Objectif :** chaque journée = un point à 24 coordonnées (puissance moyenne de 0 h à 23 h). On standardise chaque colonne : `z = (x − μ) / σ`.

**Pourquoi :** K-Means et DBSCAN reposent sur la **distance euclidienne**. Ici les 24 variables ont la même unité (kW), mais pas la même variabilité : sans standardisation, les heures de pointe (forte variance) dominent la distance et les heures de nuit comptent à peine. Standardiser donne à chaque heure le même poids (CM02, séquence 2). La section B3 teste ce que change ce choix.

In [ ]:
meta = pd.DataFrame(index=profils.index)
meta["conso_kwh"] = table_j.loc[profils.index, "conso_kwh"]
meta["jour_semaine"] = profils.index.dayofweek
meta["est_weekend"] = (profils.index.dayofweek >= 5).astype(int)
meta["mois"] = profils.index.month

scaler = StandardScaler()
Xs = scaler.fit_transform(profils)          # pas de split : clustering exploratoire
print("Forme de la matrice standardisée :", Xs.shape)
print("Moyenne par colonne (doit être ~0) :", np.round(Xs.mean(axis=0)[:3], 3), "...")
print("Écart-type par colonne (doit être 1) :", np.round(Xs.std(axis=0)[:3], 3), "...")

## B2 · PCA : résumer 24 dimensions en 2 axes
**Objectif :** la PCA trouve les axes le long desquels les journées diffèrent le plus (variance maximale), classés par importance. On regarde la **variance expliquée** (CM02, séquence 6).

**Pourquoi :** impossible de visualiser 24 dimensions ; avec 2 axes on peut *voir* les journées. La variance cumulée dit combien d'information on garde.

In [ ]:
pca_full = PCA(random_state=SEED).fit(Xs)
evr = pca_full.explained_variance_ratio_
cum = np.cumsum(evr)
n90 = int(np.argmax(cum >= 0.90) + 1)

fig, axes = plt.subplots(1, 2, figsize=(12, 4))
axes[0].bar(range(1, 9), evr[:8] * 100)
axes[0].set_title("Variance expliquée par composante (8 premières)")
axes[0].set_xlabel("Composante principale"); axes[0].set_ylabel("Variance expliquée (%)")
axes[1].plot(range(1, 25), cum * 100, marker="o")
axes[1].axhline(90, color="red", linestyle="--")
axes[1].set_title("Variance expliquée cumulée")
axes[1].set_xlabel("Nombre de composantes"); axes[1].set_ylabel("Variance cumulée (%)")
plt.tight_layout(); plt.show()

print(f"PC1 = {evr[0]:.1%} | PC2 = {evr[1]:.1%} | PC1+PC2 = {cum[1]:.1%}")
print(f"Nombre de composantes pour atteindre 90 % de variance : {n90}")

In [ ]:
# Interprétation des axes : quelles heures pèsent sur PC1 et PC2 ?
pca2 = PCA(n_components=2, random_state=SEED).fit(Xs)
X2 = pca2.transform(Xs)

fig, ax = plt.subplots(figsize=(10, 4))
ax.plot(range(24), pca2.components_[0], marker="o", label="PC1")
ax.plot(range(24), pca2.components_[1], marker="s", label="PC2")
ax.axhline(0, color="black", linewidth=0.8)
ax.set_title("Poids de chaque heure dans les deux premiers axes (chargements)")
ax.set_xlabel("Heure de la journée"); ax.set_ylabel("Poids dans la composante")
ax.legend(); plt.tight_layout(); plt.show()

**Interprétation.** Les deux premières composantes ne résument que 43,8 % de la variance (PC1 29,2 %, PC2 14,6 %), et il faut 14 composantes pour atteindre 90 %. Une fois standardisés, les profils horaires répartissent leur information sur beaucoup d'heures : la projection en 2D est utile pour visualiser, mais elle ne conserve pas l'essentiel de l'information et doit être lue avec cette réserve.

PC1 a des poids tous positifs (de 0,13 à 0,23), maximaux entre 9 h et 20 h : elle mesure le **niveau global de consommation** de la journée, surtout en journée et en soirée. PC2 oppose les heures de nuit (0 h à 5 h, poids jusqu'à 0,39) aux heures de 7 h à 23 h (poids négatifs) : elle distingue les journées à consommation nocturne relativement élevée de celles où la consommation se concentre le jour et le soir. La PCA résume mais ne regroupe pas : elle sert ici à visualiser les clusters trouvés ensuite.

## B3 · K-Means : choisir k puis décrire les groupes
**Objectif :** on lance K-Means pour k = 2 à 8 et on regarde deux indicateurs (CM02, séquence 4) :
- **Inertie** (méthode du coude) : somme des carrés des distances aux centres, qui baisse toujours quand k augmente ;
- **Silhouette** : entre −1 et 1, plus c'est haut mieux les groupes sont séparés.

**Remarque :** ces indices *guident*, ils ne décident pas seuls. Le choix final de k combine les indices **et** l'interprétation métier (taille des groupes, lisibilité).

In [ ]:
resultats = []
for k in range(2, 9):
    km = KMeans(n_clusters=k, n_init=10, random_state=SEED).fit(Xs)
    resultats.append({"k": k, "inertie": km.inertia_, "silhouette": silhouette_score(Xs, km.labels_)})
res_k = pd.DataFrame(resultats)

fig, axes = plt.subplots(1, 2, figsize=(12, 4))
axes[0].plot(res_k["k"], res_k["inertie"], marker="o")
axes[0].set_title("Méthode du coude"); axes[0].set_xlabel("Nombre de clusters k"); axes[0].set_ylabel("Inertie")
axes[1].plot(res_k["k"], res_k["silhouette"], marker="o", color="tab:green")
axes[1].set_title("Score de silhouette"); axes[1].set_xlabel("Nombre de clusters k"); axes[1].set_ylabel("Silhouette")
plt.tight_layout(); plt.show()
res_k.round(3)

**Interprétation.** L'inertie diminue de façon régulière (de 28 194 pour k = 2 à 18 864 pour k = 8) sans coude net ; une légère inflexion apparaît vers k = 3-4. La silhouette est maximale pour k = 2 (0,203) puis décroît (0,162 pour k = 3, 0,134 pour k = 4, environ 0,09 pour k = 7 et 8). Les deux indices sont peu tranchés et les valeurs de silhouette sont faibles : les journées forment plutôt un continuum que des groupes nettement séparés.

Nous retenons k = 2, qui maximise la silhouette et donne deux groupes lisibles pour le métier (section B4) ; avec k plus grand, la silhouette baisse, c'est-à-dire que les groupes sont moins séparés. C'est une simplification à assumer dans le rapport.

Les deux groupes comptent 913 et 504 jours. L'ARI de 0,59 avec un K-Means sans standardisation montre que la standardisation modifie sensiblement le regroupement (sans elle, les heures de pointe dominent la distance) : nous la conservons pour donner le même poids à chaque heure.

In [ ]:
K_CHOISI = int(res_k.loc[res_k["silhouette"].idxmax(), "k"])      # par défaut : le k de meilleure silhouette
print("k retenu :", K_CHOISI)

km = KMeans(n_clusters=K_CHOISI, n_init=10, random_state=SEED).fit(Xs)
labels_km = km.labels_
meta["cluster_km"] = labels_km

taille = pd.Series(labels_km).value_counts().sort_index()
print("Taille des clusters (nombre de jours) :")
print(taille)

# Sensibilité au choix de standardiser : le résultat change-t-il sans standardisation ?
km_brut = KMeans(n_clusters=K_CHOISI, n_init=10, random_state=SEED).fit(profils.values)
ari = adjusted_rand_score(labels_km, km_brut.labels_)
print(f"Accord avec un K-Means sans standardisation (ARI, 1 = identique) : {ari:.2f}")

**Lecture de l'ARI :** 1 = les deux regroupements sont identiques, proche de 0 = ils n'ont presque rien en commun. Si l'ARI est bas, **la standardisation change vraiment le résultat** : c'est un choix méthodologique à assumer et à expliquer.

## B4 · Décrire les profils de journées
**Objectif :** pour chaque cluster, tracer la **courbe moyenne de 24 h** (en kW, pas en valeurs standardisées, pour que ce soit lisible) et résumer : taille, consommation moyenne, part de week-ends, mois dominants.

**Pourquoi :** nommer un cluster (« journée d'hiver », « journée d'absence ») est une **interprétation humaine après coup** (CM02, séquence 7). Elle doit s'appuyer sur ces chiffres, pas sur l'intuition seule.

In [ ]:
# Courbes moyennes par cluster
courbes = profils.groupby(labels_km).mean()
fig, ax = plt.subplots(figsize=(10, 4.5))
for c in courbes.index:
    ax.plot(range(24), courbes.loc[c].values, marker="o", label=f"Cluster {c} (n = {int(taille[c])})")
ax.set_title("Profil horaire moyen de chaque cluster")
ax.set_xlabel("Heure de la journée"); ax.set_ylabel("Puissance moyenne (kW)")
ax.legend(); plt.tight_layout(); plt.show()

# Résumé chiffré par cluster
resume = meta.groupby("cluster_km").agg(
    jours=("conso_kwh", "size"),
    conso_moy_kwh=("conso_kwh", "mean"),
    part_weekend=("est_weekend", "mean"),
).round(2)
resume["mois_dominant"] = meta.groupby("cluster_km")["mois"].agg(lambda s: int(s.mode().iloc[0]))
resume

In [ ]:
# Répartition des clusters par mois (en % de chaque cluster) : lien avec la saison
rep_mois = pd.crosstab(meta["cluster_km"], meta["mois"], normalize="index").mul(100).round(0).astype(int)
print("Part (%) de chaque cluster dans chaque mois (lignes = cluster, colonnes = mois) :")
print(rep_mois)

# Visualisation des journées dans le plan PCA, colorées par cluster
fig, ax = plt.subplots(figsize=(8, 6))
sc = ax.scatter(X2[:, 0], X2[:, 1], c=labels_km, cmap="tab10", s=12, alpha=0.7)
ax.set_title("Journées dans le plan des deux premiers axes de la PCA")
ax.set_xlabel(f"PC1 ({evr[0]:.0%} de la variance)"); ax.set_ylabel(f"PC2 ({evr[1]:.0%} de la variance)")
ax.legend(*sc.legend_elements(), title="Cluster")
plt.tight_layout(); plt.show()

**Interprétation.** Le groupe 0 (913 jours, 64 % des journées) consomme en moyenne 20,6 kWh/jour, compte 19 % de week-ends et a juillet pour mois dominant. Sa courbe montre un pic le matin (environ 1,3 kW vers 7 h), une décroissance régulière en journée jusqu'à environ 0,5 kW vers 16 h, puis un pic modéré le soir (environ 1,35 kW) : un schéma de journée ordinaire où le foyer est probablement peu présent en journée.

Le groupe 1 (504 jours, 36 %) consomme en moyenne 36,3 kWh/jour, avec 44 % de week-ends et janvier pour mois dominant. Sa courbe garde un plateau élevé toute la journée (environ 1,5 à 1,7 kW de 7 h à 15 h), un niveau nocturne plus haut et un pic marqué le soir (environ 2,4 kW vers 20 h) : des journées de présence et de forte consommation.

Le tableau par mois confirme le lien avec la saison : 70 % des journées du groupe 1 tombent entre novembre et mars (17 % en janvier, 16 % en décembre), et le groupe est quasi absent en juillet et en août (0 % et 1 %). Le groupe 0 est réparti sur toute l'année avec un maximum en été (13 % en juillet, 12 % en août).

Dans le plan de la PCA, les deux groupes se touchent autour de PC1 ≈ 0 à 1, sans frontière nette : le groupe 1 s'étire vers la droite et le haut, avec quelques journées très éloignées du nuage (PC1 supérieur à 7). Cette continuité est cohérente avec la silhouette faible (0,203) : il s'agit d'un continuum de journées plus que de deux populations distinctes.

Les deux groupes ont la même forme générale (pic du matin, pic du soir) et diffèrent surtout par le niveau et par le plateau diurne : le clustering retrouve donc en grande partie la saison et le week-end, ce qui est cohérent avec l'analyse exploratoire mais n'est pas une découverte surprenante. Les noms que nous leur donnons (« journées ordinaires, plutôt estivales » et « journées de présence, plutôt hivernales ») sont une interprétation après coup, à valider avec des informations extérieures aux données (météo, calendrier de présence). Appartenir au même groupe ne prouve pas une cause commune.

## B5 · DBSCAN : repérer les journées atypiques
**Objectif :** DBSCAN regroupe les points situés dans des zones **denses** et isole les autres comme **bruit** (label −1). Contrairement à K-Means, il ne force aucun point dans un cluster (CM02, séquence 5).

**Deux réglages :**
- `min_samples` : nombre minimal de journées voisines pour qu'une journée soit « centrale » ;
- `eps` (ε) : rayon de voisinage. On le choisit avec le **graphique des k plus proches voisins** : on trie la distance de chaque journée à son `min_samples`-ième voisin ; le « coude » donne un ε raisonnable.

**Lecture :** une journée « bruit » n'est **pas une erreur de mesure** : c'est une journée qui ne ressemble à aucune zone dense (CM02, question 07), donc un signal à investiguer.

In [ ]:
MIN_SAMPLES = 10
dist, _ = NearestNeighbors(n_neighbors=MIN_SAMPLES).fit(Xs).kneighbors(Xs)
kdist = np.sort(dist[:, -1])

fig, ax = plt.subplots(figsize=(8, 4))
ax.plot(kdist)
ax.set_title(f"Distance au {MIN_SAMPLES}e plus proche voisin (triée)")
ax.set_xlabel("Journées triées"); ax.set_ylabel("Distance (espace standardisé)")
plt.tight_layout(); plt.show()

# Sensibilité à epsilon : combien de clusters et de journées « bruit » pour chaque valeur ?
lignes = []
for p in (50, 70, 80, 90, 95):
    e = float(np.percentile(kdist, p))
    lab = DBSCAN(eps=e, min_samples=MIN_SAMPLES).fit_predict(Xs)
    lignes.append({"percentile": p, "eps": round(e, 2),
                   "clusters": len(set(lab)) - (1 if -1 in lab else 0),
                   "jours_bruit": int((lab == -1).sum())})
pd.DataFrame(lignes)

**Interprétation.** La courbe des distances au 10e plus proche voisin est presque plate (environ 2 à 4,5) pour la grande majorité des journées, puis monte nettement pour les quelque 100 dernières (jusqu'à environ 11) : il existe un noyau dense de journées semblables et une minorité de journées isolées. Il n'y a pas de coude net qui désigne une seule valeur de ε, mais le décrochage se situe autour de 5.

Quelle que soit la valeur de ε testée (de 3,09 à 5,62), DBSCAN ne trouve qu'**un seul cluster** ; seul le nombre de journées « bruit » change (512 pour ε = 3,09, 237 pour 3,74, 161 pour 4,14, 71 pour 5,02 et 34 pour 5,62). C'est l'effet de la dimension élevée (24 heures) : les journées forment un nuage continu, sans zones denses séparées par du vide. DBSCAN ne segmente donc pas ici les journées : il fonctionne comme un **détecteur de journées atypiques**, et c'est ce que nous lui demandons. Nous retenons ε = 5,02 (90e centile de la courbe) avec `min_samples = 10`, ce qui isole environ 5 % des journées.

In [ ]:
EPS = round(float(np.percentile(kdist, 90)), 2)          # par défaut : 90e centile de la courbe
db = DBSCAN(eps=EPS, min_samples=MIN_SAMPLES).fit(Xs)
labels_db = db.labels_
meta["cluster_db"] = labels_db

n_clusters_db = len(set(labels_db)) - (1 if -1 in labels_db else 0)
print(f"eps = {EPS} | min_samples = {MIN_SAMPLES}")
print(f"Clusters trouvés : {n_clusters_db} | journées « bruit » : {int((labels_db == -1).sum())} "
      f"({(labels_db == -1).mean():.1%})")

print("\nCroisement K-Means (lignes) x DBSCAN (colonnes, -1 = bruit) :")
print(pd.crosstab(meta["cluster_km"], meta["cluster_db"]))

bruit = meta[meta["cluster_db"] == -1][["conso_kwh", "jour_semaine", "mois"]].sort_values("conso_kwh")
print("\nJournées atypiques selon DBSCAN (triées par consommation) :")
bruit.round(1).head(20)

**Interprétation.** Avec ε = 5,02 et `min_samples = 10`, DBSCAN trouve 1 cluster et 71 journées « bruit » (5,0 %). Le croisement avec K-Means montre que 63 de ces 71 journées appartiennent au groupe 1 (forte consommation) et 8 seulement au groupe 0 : les journées atypiques se trouvent presque toutes dans le groupe le plus dispersé.

Le tableau est trié par consommation croissante : les 20 lignes affichées sont donc les moins consommatrices des 71 et vont de 19,4 à 39,9 kWh. Les 51 autres dépassent 39,9 kWh, pour une moyenne générale de 26,2 kWh. Les journées atypiques sont donc surtout des journées de forte consommation, ou des journées de consommation moyenne dont la **forme** horaire sort de l'ordinaire. Parmi les 20 affichées, 13 sont des dimanches (contre une journée sur sept en moyenne) et 2 des samedis, et 11 tombent en octobre, novembre ou avril, périodes de transition entre saisons (hypothèse : début et fin de la période de chauffage).

Aucune journée de très faible consommation n'apparaît parmi les atypiques : les creux d'absence (5 à 9 kWh, notebook 01) ne sont pas du bruit, probablement parce qu'ils forment eux-mêmes un groupe dense de journées semblables.

Rappel : une anomalie est un signal statistique à investiguer, pas la preuve d'une erreur de mesure ni d'une cause.

## B6 · Lien entre les deux problématiques
**Question :** *quels types de journées le modèle de prévision rate-t-il le plus ?* C'est la phrase de liaison du cadrage.

**Méthode :** on croise les **erreurs hors échantillon** de la partie A (jours de validation, test non touché) avec les clusters de la partie B.

> **Précaution :** le cluster d'un jour *J* est calculé avec les mesures du jour *J*. On ne peut donc **pas** l'utiliser comme variable de prévision de *J* (ce serait une fuite). Il sert ici uniquement à **expliquer** les erreurs après coup, ou à segmenter l'analyse.

In [ ]:
lien = err.copy()
lien["cluster_km"] = meta["cluster_km"].reindex(lien.index)
lien["cluster_db"] = meta["cluster_db"].reindex(lien.index)

print("Erreur de prévision par cluster K-Means (jours de validation) :")
par_cluster = lien.groupby("cluster_km").agg(jours=("abs_rf", "size"),
                                              MAE_rf=("abs_rf", "mean"),
                                              MAE_baseline_lag7=("abs_lag7", "mean")).round(2)
print(par_cluster)

lien["atypique_db"] = (lien["cluster_db"] == -1)
print("\nErreur de prévision : journées atypiques (DBSCAN) vs journées normales :")
print(lien.groupby("atypique_db").agg(jours=("abs_rf", "size"),
                                      MAE_rf=("abs_rf", "mean"),
                                      MAE_baseline_lag7=("abs_lag7", "mean")).round(2))

**Interprétation.** En valeur absolue, le modèle se trompe moins sur les journées du groupe 0 (MAE 4,74 kWh/jour sur 594 jours de validation) que sur celles du groupe 1 (6,15 sur 316 jours). Cette comparaison est trompeuse : le groupe 1 consomme beaucoup plus (36,3 contre 20,6 kWh en moyenne). Rapportée à la consommation moyenne du groupe, l'erreur est d'environ 17 % pour le groupe 1 contre 23 % pour le groupe 0 : les journées de forte consommation sont plutôt mieux prévues en relatif. Dans les deux groupes le Random Forest bat la baseline « semaine dernière » (5,57 → 4,74 pour le groupe 0 ; 7,96 → 6,15 pour le groupe 1).

Les journées atypiques selon DBSCAN sont nettement plus difficiles à prévoir : MAE de 10,26 kWh/jour sur 35 jours, contre 5,03 sur les 875 autres, soit environ deux fois plus. Elles ne représentent que 3,8 % des jours de validation mais environ 7,5 % de l'erreur totale. Le modèle reste meilleur que la baseline sur ces jours (10,26 contre 11,55), mais l'écart est faible (environ 11 %, contre environ 19 % sur les jours normaux).

Conséquence métier : les erreurs se concentrent sur des journées qui sortent de la norme, dont le profil n'est connu qu'après coup ; le cluster ne peut donc pas servir à prévoir. En pratique, on peut prévoir une marge de sécurité plus large pour les périodes où ces journées semblent fréquentes (dimanches, intersaisons), ce que le notebook 03 chiffrera en euros.

## B7 · Sauvegarde et bilan

In [ ]:
meta[["cluster_km", "cluster_db"]].to_csv(os.path.join(DATA_DIR, "clusters_journaliers.csv"), index_label="date")
print("Fichiers produits dans", DATA_DIR + "/ :")
print(sorted(f for f in os.listdir(DATA_DIR) if f.endswith(".csv")))

### Résultats principaux
**Problématique 1 (prévision)**
- Validation croisée temporelle (5 plis) : meilleure baseline « comme hier » 6,18 ± 0,69 kWh/jour ; régression linéaire 5,32 ± 0,95 ; arbre sans limite 8,22 ± 1,17 (surapprentissage, erreur nulle sur le train) ; Random Forest réglée (`max_depth = 12`, `min_samples_leaf = 10`) 5,23 ± 0,60, soit 15,4 % de gain.
- Test 2010 (305 jours, utilisé une seule fois) : MAE de 4,22 kWh/jour contre 5,10 pour « comme hier » et 5,98 pour « semaine dernière », soit **17,2 % de gain : critère de succès atteint**. RMSE 5,78 ; intervalle à 90 % de −10,3 à +10,6 kWh, couvrant 92,5 % des jours de test.
- Erreurs : plus fortes le week-end et en février, novembre et août ; les plus grosses sont des ruptures brutales (départs, retours) difficiles à anticiper.

**Problématique 2 (non supervisé)**
- PCA : 43,8 % de la variance sur 2 axes, 14 composantes pour 90 %.
- K-Means (k = 2, silhouette 0,203 : structure faible) : 913 journées ordinaires (20,6 kWh, 19 % de week-ends, plutôt estivales) et 504 journées de forte consommation (36,3 kWh, 44 % de week-ends, 70 % entre novembre et mars).
- DBSCAN (ε = 5,02, `min_samples = 10`) : un seul cluster et 71 journées atypiques (5,0 %), concentrées dans le groupe de forte consommation et souvent des dimanches.

**Lien entre les deux problématiques :** les journées atypiques sont deux fois plus mal prévues (MAE 10,26 contre 5,03) et pèsent environ 7,5 % de l'erreur pour 3,8 % des jours.

### Synthèse de la démarche
**Problématique 1 (prévision)**
- Split chronologique (train jusqu'à fin 2009, test sur 2010), `TimeSeriesSplit` à 5 plis.
- Comparaison de 2 baselines, d'une régression linéaire, d'un arbre et d'une forêt aléatoire réglée (MAE train, MAE validation, écart-type).
- Diagnostic : surapprentissage de l'arbre, importance des variables, erreurs par jour et par mois, pires jours.
- Évaluation finale sur le test, effectuée une seule fois : MAE, RMSE, gain par rapport à la baseline, critère de succès, intervalle à 90 %.
- Métrique principale : MAE (kWh/jour) ; garde-fou : RMSE ; référence : gain par rapport à la baseline.

**Problématique 2 (non supervisé)**
- Standardisation, PCA (variance expliquée, interprétation des axes), choix de k (coude, silhouette, interprétation), DBSCAN (choix de ε, journées atypiques).
- Description de chaque profil, avec ses limites : un cluster n'est pas une catégorie réelle, corrélation n'est pas causalité, un seul foyer.
- Lien avec la prévision : erreurs par type de journée.

### Suite — notebook 03 (décision et ROI)
Conversion de la MAE et des erreurs en euros (hypothèses de coût explicites pour le kWh sur-acheté et sous-acheté), comparaison avec la baseline, calcul du ROI avec scénarios pessimiste, central et optimiste, analyse de sensibilité, puis registre de risques.